In [1]:
import sys
sys.path.append("../")
from tools import getPairInfo, metric
import matplotlib.pyplot as plt
import numpy as np
import torch
import os
import re
import pandas as pd
from matplotlib import rc

In [2]:
# 选择预测器版本
from predictor_iter import predict
# MACCSKeys
modelPath = {
    'kcat_8':"../../IterTransferLearning/KCATModel/confirm-MACCS#openKKmInfer#openLog10-KCAT-train-8/model/pre-train-trainR2:0.9408-devR2=0.7459-RMSE=0.7751-MAE=0.4787-epoch=3",
    'kcat_1':"../../IterTransferLearning/KCATModel/confirm-MACCS#openKKmInfer#openLog10-KCAT-train-1/model/pre-train-trainR2:0.9166-devR2=0.6752-RMSE=0.8763-MAE=0.5617-epoch=94",
    'km_8':"../../IterTransferLearning/KMModel/confirm-MACCS#openKKmInfer#openLog10-KM-train-8/model/pre-train-trainR2:0.9303-devR2=0.7163-RMSE=0.6960-MAE=0.4802-epoch=3",
    'km_1':"../../IterTransferLearning/KMModel/confirm-MACCS#openKKmInfer#openLog10-KM-train-1/model/pre-train-trainR2:0.8980-devR2=0.6307-RMSE=0.7941-MAE=0.5537-epoch=100"
}

def err(x, y):
    x = np.array(x)
    y = np.array(y)
    return np.abs(x-y)

results = {'kcat':[], 'km':[]}
AEData = {'kcat':[], 'km':[]}
all_kinetics_data = torch.load("../../KCAT/Data/all_kinetics_data.pt")
# 设定
# KineticsType = 'kcat'
for KineticsType in ['kcat', 'km']:
    # 加载kcat或Km测试集
    TestPairInfo = torch.load(f"../Data/{KineticsType.upper()}TestPairInfo")
    # 加载索引数据
    indexSeq = torch.load(f"../../{KineticsType.upper()}/Data/NewestFeature/index_seq")
    indexSmiles = torch.load(f"../../{KineticsType.upper()}/Data/NewestFeature/index_smiles")
    # indexSet
    indexSet1 = all_kinetics_data['index'][f'{KineticsType}_kkm_IndexSet']
    indexSet2 = all_kinetics_data['index']['kcat_km_IndexSet']
    # 从Kcat或km测试集中去除存在于kkm训练集中的样本
    pureTestPairInfo = []
    for item in TestPairInfo:
        seq = indexSeq[item[0]]
        smiles = indexSmiles[item[1]]
        if f"{seq}_{smiles}" not in indexSet1 and f"{seq}_{smiles}" not in indexSet2:
            pureTestPairInfo.append(item)

    
    iter1Real, iter1Pre = predict(
                    KineticsType.upper(),
                    modelPath[KineticsType+'_1'],
                    pureTestPairInfo,
                    f"../../{KineticsType.upper()}/Data/NewestFeature/esm1v_t33_650M_UR90S_1_embeding_1280/",
                    f"../../{KineticsType.upper()}/Data/NewestFeature/index_smiles",
                    True,
                    torch.device("cuda:1"),
                    "MACCSKeys"
            )
    res = metric(iter1Pre, iter1Real, True)
    res['type'] = 'iter1'
    results[KineticsType].append(res)
    iter8Real, iter8Pre = predict(
                    KineticsType.upper(),
                    modelPath[KineticsType+'_8'],
                    pureTestPairInfo,
                    f"../../{KineticsType.upper()}/Data/NewestFeature/esm1v_t33_650M_UR90S_1_embeding_1280/",
                    f"../../{KineticsType.upper()}/Data/NewestFeature/index_smiles",
                    True,
                    torch.device("cuda:1"),
                    "MACCSKeys"
            )
    res = metric(iter8Pre, iter8Real, True)
    res['type'] = 'iter8'
    results[KineticsType].append(res)
    
    AEData[KineticsType].append(err(iter1Pre, iter1Real))
    AEData[KineticsType].append(err(iter8Pre, iter8Real))
#     results = pd.DataFrame(results)
#     results
AEData = pd.DataFrame(AEData)
AEData.to_csv("../Fig/Fig6A-MACCSKeys-AEData.csv")
for k, v in results.items():
    print(k)
    print(pd.DataFrame(v))

log10:True molType:MACCSKeys
F


100%|█████████████████████████████████████████████████████████████| 28/28 [00:08<00:00,  3.48it/s]


log10:True molType:MACCSKeys
F


100%|█████████████████████████████████████████████████████████████| 28/28 [00:06<00:00,  4.42it/s]


log10:True molType:MACCSKeys
F


100%|█████████████████████████████████████████████████████████████| 21/21 [00:05<00:00,  3.75it/s]


log10:True molType:MACCSKeys
F


100%|█████████████████████████████████████████████████████████████| 21/21 [00:05<00:00,  3.67it/s]

kcat
        MAE      rmse        r2         r  p_value  samples   type
0  0.561968  0.874451  0.671829  0.822974      0.0     2765  iter1
1  0.495740  0.805371  0.721630  0.851842      0.0     2765  iter8
km
        MAE      rmse        r2         r  p_value  samples   type
0  0.559901  0.791806  0.626790  0.797039      0.0     2099  iter1
1  0.497113  0.719434  0.691896  0.834744      0.0     2099  iter8


In [3]:
import scipy.stats as stats

In [4]:
stats.wilcoxon(AEData['kcat'][0],AEData['kcat'][1], alternative='two-sided')

WilcoxonResult(statistic=1495638.5, pvalue=3.497269706502023e-23)

In [5]:
stats.wilcoxon(AEData['km'][0],AEData['km'][1], alternative='two-sided')

WilcoxonResult(statistic=822847.5, pvalue=9.072288437524653e-24)